In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers accelerate bitsandbytes sentencepiece faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.9 MB/s eta 0:00:00


In [4]:
import os
import re
import json
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"
EVALUATION_PATH = PROJECT_PATH / "evaluation"

print(PROJECT_PATH)

/content/drive/MyDrive/uterine-emg-rag


In [5]:
import json

with open(
    EVALUATION_PATH / "gold_queries.json",
    "r",
    encoding="utf-8"
) as f:
    gold_queries = json.load(f)

print("Number of queries:", len(gold_queries))
print("\nFirst query:")
print(json.dumps(gold_queries[0], indent=2, ensure_ascii=False))

Number of queries: 30

First query:
{
  "query_id": "Q001",
  "question": "What are the main characteristics of uterine electromyography (EMG) signals?",
  "category": "signal_characteristics",
  "relevant_paper_ids": [
    "paper1",
    "paper2",
    "paper3",
    "paper4",
    "paper5"
  ]
}


In [6]:
for i, item in enumerate(gold_queries[:5]):
    print(f"\n{'='*70}")
    print(f"Query {i+1}")
    print(f"{'='*70}")
    print("Question:", item["question"])
    print("Relevant paper IDs:", item.get("relevant_paper_ids"))


Query 1
Question: What are the main characteristics of uterine electromyography (EMG) signals?
Relevant paper IDs: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']

Query 2
Question: What physiological activity generates uterine EMG signals?
Relevant paper IDs: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']

Query 3
Question: How do uterine EMG signals change as pregnancy progresses?
Relevant paper IDs: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']

Query 4
Question: How does uterine electrical activity change during the transition from pregnancy to labor?
Relevant paper IDs: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']

Query 5
Question: What differences in uterine EMG activity have been observed between laboring and non-laboring women?
Relevant paper IDs: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']


In [7]:
print("FAISS directory contents:")

for p in FAISS_PATH.iterdir():
    print(p.name)

FAISS directory contents:
uterine_emg.index


In [8]:
print("Embeddings directory:")

for p in EMBEDDINGS_PATH.iterdir():
    print(p.name)

Embeddings directory:
embeddings.npy
chunk_metadata.csv


In [9]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [10]:
print("Embedding dimension:",
      embedder.get_sentence_embedding_dimension())

Embedding dimension: 384


/tmp/ipykernel_2035/1839089237.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedder.get_sentence_embedding_dimension())


In [11]:
index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

print("FAISS index loaded.")
print("Number of vectors:", index.ntotal)
print("Dimension:", index.d)

FAISS index loaded.
Number of vectors: 833
Dimension: 384


In [12]:
metadata_df = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

print("Shape:", metadata_df.shape)
print("\nColumns:")
print(metadata_df.columns.tolist())

print("\nFirst 5 rows:")
display(metadata_df.head())

Shape: (833, 4)

Columns:
['paper', 'chunk_id', 'text', 'characters']

First 5 rows:


,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966


In [13]:
print(metadata_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   paper       833 non-null    object
 1   chunk_id    833 non-null    int64 
 2   text        833 non-null    object
 3   characters  833 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 26.2+ KB
None


In [14]:
for col in metadata_df.columns:
    print(f"\nColumn: {col}")
    print(metadata_df[col].head(3).tolist())


Column: paper
['paper1', 'paper1', 'paper1']

Column: chunk_id
[0, 1, 2]

Column: text
['Contents lists available at ScienceDirect\nArtificial Intelligence In Medicine\njournal homepage: www.elsevier.com/locate/artmed\n \nElectrohysterography in modern obstetrics: Advances in signal processing, \nmachine learning, and clinical applications\nKaterina Barnova a,b\n, Radek Martinek a\n, Jitka Horakova c, Ondrej Simetka c\n, \nRadana Vilimkova Kahankova a\n,∗\na VSB – Technical University of Ostrava, Department of Cybernetics and Biomedical Engineering, 17. listopadu 2172/15, Ostrava, 70800, Czechia\nb Hospital AGEL Trinec-Podlesi, Telemedicine Center, Konska 453, Trinec, 739 61, Czechia\nc University Hospital Ostrava, Department of Obstetrics and Gynecology, 17. listopadu 1790/5, Ostrava, 708 00, Czechia\nA R T I C L E  I N F O\nKeywords:\nUterine electromyography\nUterine activity monitoring\nEHG signal processing\nUterine contractions detection\nTerm/preterm birth prediction\nPregnancy

In [15]:
print("FAISS vectors:", index.ntotal)
print("Metadata rows:", len(metadata_df))

FAISS vectors: 833
Metadata rows: 833


In [16]:
embeddings = np.load(
    EMBEDDINGS_PATH / "embeddings.npy"
)

print("Embeddings shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)
print("FAISS dimension:", index.d)

Embeddings shape: (833, 384)
FAISS vectors: 833
FAISS dimension: 384


In [17]:
print("Row 0 metadata:")
display(metadata_df.iloc[0])

print("\nRow 1 metadata:")
display(metadata_df.iloc[1])

print("\nRow 2 metadata:")
display(metadata_df.iloc[2])

Row 0 metadata:


,0
paper,paper1
chunk_id,0
text,Contents lists available at ScienceDirect\nArt...
characters,927



Row 1 metadata:


,1
paper,paper1
chunk_id,1
text,Keywords:\nUterine electromyography\nUterine a...
characters,946



Row 2 metadata:


,2
paper,paper1
chunk_id,2
text,"transform, and for data classification, such a..."
characters,996


In [18]:
print("Embedding 0 shape:", embeddings[0].shape)
print("Embedding 1 shape:", embeddings[1].shape)
print("Embedding 2 shape:", embeddings[2].shape)

Embedding 0 shape: (384,)
Embedding 1 shape: (384,)
Embedding 2 shape: (384,)


In [19]:
CHUNKS_PATH = PROJECT_PATH / "processed"

chunks_df = pd.read_csv(
    CHUNKS_PATH / "chunks.csv"
)

print("Shape:", chunks_df.shape)

print("\nColumns:")
print(chunks_df.columns.tolist())

display(chunks_df.head())

Shape: (833, 4)

Columns:
['paper', 'chunk_id', 'text', 'characters']


,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966


In [20]:
print(chunks_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   paper       833 non-null    object
 1   chunk_id    833 non-null    int64 
 2   text        833 non-null    object
 3   characters  833 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 26.2+ KB
None


In [21]:
print("Metadata columns:")
print(metadata_df.columns.tolist())

print("\nChunks columns:")
print(chunks_df.columns.tolist())

Metadata columns:
['paper', 'chunk_id', 'text', 'characters']

Chunks columns:
['paper', 'chunk_id', 'text', 'characters']


In [22]:
with open(
    CHUNKS_PATH / "chunks.json",
    "r",
    encoding="utf-8"
) as f:
    chunks_json = json.load(f)

print("Type:", type(chunks_json))

if isinstance(chunks_json, list):
    print("Number of chunks:", len(chunks_json))
    print("\nFirst chunk:")
    print(
        json.dumps(
            chunks_json[0],
            indent=2,
            ensure_ascii=False
        )
    )

elif isinstance(chunks_json, dict):
    print("Keys:")
    print(chunks_json.keys())

Type: <class 'list'>
Number of chunks: 833

First chunk:
{
  "paper": "paper1",
  "chunk_id": 0,
  "text": "Contents lists available at ScienceDirect\nArtificial Intelligence In Medicine\njournal homepage: www.elsevier.com/locate/artmed\n \nElectrohysterography in modern obstetrics: Advances in signal processing, \nmachine learning, and clinical applications\nKaterina Barnova a,b\n, Radek Martinek a\n, Jitka Horakova c, Ondrej Simetka c\n, \nRadana Vilimkova Kahankova a\n,∗\na VSB – Technical University of Ostrava, Department of Cybernetics and Biomedical Engineering, 17. listopadu 2172/15, Ostrava, 70800, Czechia\nb Hospital AGEL Trinec-Podlesi, Telemedicine Center, Konska 453, Trinec, 739 61, Czechia\nc University Hospital Ostrava, Department of Obstetrics and Gynecology, 17. listopadu 1790/5, Ostrava, 708 00, Czechia\nA R T I C L E  I N F O\nKeywords:\nUterine electromyography\nUterine activity monitoring\nEHG signal processing\nUterine contractions detection\nTerm/preterm birth pre

In [23]:
def retrieve(
    query,
    top_k=5
):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        row = metadata_df.iloc[idx]

        results.append({
            "rank": rank,
            "score": float(score),
            "faiss_index": int(idx),
            "paper_id": row["paper"],
            "chunk_id": row["chunk_id"]
        })

    return results

In [24]:
query = gold_queries[0]["question"]

results = retrieve(
    query,
    top_k=5
)

for result in results:
    print(result)

{'rank': 1, 'score': 0.7830531597137451, 'faiss_index': 297, 'paper_id': 'paper2', 'chunk_id': np.int64(40)}
{'rank': 2, 'score': 0.7452520728111267, 'faiss_index': 301, 'paper_id': 'paper2', 'chunk_id': np.int64(44)}
{'rank': 3, 'score': 0.7357030510902405, 'faiss_index': 311, 'paper_id': 'paper2', 'chunk_id': np.int64(54)}
{'rank': 4, 'score': 0.729323148727417, 'faiss_index': 309, 'paper_id': 'paper2', 'chunk_id': np.int64(52)}
{'rank': 5, 'score': 0.7284902334213257, 'faiss_index': 306, 'paper_id': 'paper2', 'chunk_id': np.int64(49)}


In [25]:
# ==========================================
# Inspect metadata and chunk structures
# ==========================================

# 1. Metadata
metadata_df = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

print("=" * 80)
print("CHUNK METADATA")
print("=" * 80)

print("Shape:", metadata_df.shape)
print("Columns:", metadata_df.columns.tolist())

display(metadata_df.head())

# 2. Embeddings
embeddings = np.load(
    EMBEDDINGS_PATH / "embeddings.npy"
)

print("\n" + "=" * 80)
print("EMBEDDINGS")
print("=" * 80)

print("Shape:", embeddings.shape)

# 3. FAISS
print("\n" + "=" * 80)
print("FAISS")
print("=" * 80)

print("Vectors:", index.ntotal)
print("Dimension:", index.d)

# 4. Processed chunks CSV
chunks_df = pd.read_csv(
    PROJECT_PATH / "processed" / "chunks.csv"
)

print("\n" + "=" * 80)
print("CHUNKS CSV")
print("=" * 80)

print("Shape:", chunks_df.shape)
print("Columns:", chunks_df.columns.tolist())

display(chunks_df.head())

# 5. chunks.json
with open(
    PROJECT_PATH / "processed" / "chunks.json",
    "r",
    encoding="utf-8"
) as f:
    chunks_json = json.load(f)

print("\n" + "=" * 80)
print("CHUNKS JSON")
print("=" * 80)

print("Type:", type(chunks_json))

if isinstance(chunks_json, list):
    print("Number of items:", len(chunks_json))
    print("\nFirst item:")
    print(
        json.dumps(
            chunks_json[0],
            indent=2,
            ensure_ascii=False
        )
    )

elif isinstance(chunks_json, dict):
    print("Keys:", list(chunks_json.keys()))

CHUNK METADATA
Shape: (833, 4)
Columns: ['paper', 'chunk_id', 'text', 'characters']


,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966



EMBEDDINGS
Shape: (833, 384)

FAISS
Vectors: 833
Dimension: 384

CHUNKS CSV
Shape: (833, 4)
Columns: ['paper', 'chunk_id', 'text', 'characters']


,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966



CHUNKS JSON
Type: <class 'list'>
Number of items: 833

First item:
{
  "paper": "paper1",
  "chunk_id": 0,
  "text": "Contents lists available at ScienceDirect\nArtificial Intelligence In Medicine\njournal homepage: www.elsevier.com/locate/artmed\n \nElectrohysterography in modern obstetrics: Advances in signal processing, \nmachine learning, and clinical applications\nKaterina Barnova a,b\n, Radek Martinek a\n, Jitka Horakova c, Ondrej Simetka c\n, \nRadana Vilimkova Kahankova a\n,∗\na VSB – Technical University of Ostrava, Department of Cybernetics and Biomedical Engineering, 17. listopadu 2172/15, Ostrava, 70800, Czechia\nb Hospital AGEL Trinec-Podlesi, Telemedicine Center, Konska 453, Trinec, 739 61, Czechia\nc University Hospital Ostrava, Department of Obstetrics and Gynecology, 17. listopadu 1790/5, Ostrava, 708 00, Czechia\nA R T I C L E  I N F O\nKeywords:\nUterine electromyography\nUterine activity monitoring\nEHG signal processing\nUterine contractions detection\nTerm/preter

In [26]:
print("FAISS vectors :", index.ntotal)
print("Embeddings    :", len(embeddings))
print("Metadata rows :", len(metadata_df))
print("Chunks rows   :", len(chunks_df))
print("Chunks JSON   :", len(chunks_json))

FAISS vectors : 833
Embeddings    : 833
Metadata rows : 833
Chunks rows   : 833
Chunks JSON   : 833


In [27]:
for idx in [0, 1, 2, 100, 500, 832]:

    row = metadata_df.iloc[idx]

    print("=" * 70)
    print("FAISS index:", idx)
    print("Paper:", row["paper"])
    print("Chunk ID:", row["chunk_id"])
    print("Characters:", row["characters"])
    print("Text:", row["text"][:150].replace("\n", " "))

FAISS index: 0
Paper: paper1
Chunk ID: 0
Characters: 927
Text: Contents lists available at ScienceDirect Artificial Intelligence In Medicine journal homepage: www.elsevier.com/locate/artmed   Electrohysterography 
FAISS index: 1
Paper: paper1
Chunk ID: 1
Characters: 946
Text: Keywords: Uterine electromyography Uterine activity monitoring EHG signal processing Uterine contractions detection Term/preterm birth prediction Preg
FAISS index: 2
Paper: paper1
Chunk ID: 2
Characters: 996
Text: transform, and for data classification, such as neural networks or support vector machine, highlighting their  performance and limitations. Despite si
FAISS index: 100
Paper: paper1
Chunk ID: 100
Characters: 951
Text: 4.5.4. Convolution neural network for preterm birth prediction Convolution neural network has proven to be a universal approach  suitable for both ute
FAISS index: 500
Paper: paper5
Chunk ID: 21
Characters: 903
Text: The average number of multiplications per sample (MPS) is used as a measur

In [28]:
def retrieve(
    query,
    top_k=5
):
    """
    Retrieve top-k chunks using MiniLM + FAISS.
    """

    # 1. Encode query
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # 2. FAISS search
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    # 3. Convert FAISS results to readable records
    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx == -1:
            continue

        row = metadata_df.iloc[idx]

        results.append({
            "rank": rank,
            "faiss_index": int(idx),
            "score": float(score),
            "paper": row["paper"],
            "chunk_id": int(row["chunk_id"]),
            "text": row["text"]
        })

    return results

In [29]:
query = gold_queries[0]["question"]

print("QUERY:")
print(query)

results = retrieve(
    query,
    top_k=5
)

QUERY:
What are the main characteristics of uterine electromyography (EMG) signals?


In [30]:
for result in results:

    print("\n" + "=" * 80)

    print("Rank       :", result["rank"])
    print("Score      :", result["score"])
    print("FAISS index:", result["faiss_index"])
    print("Paper      :", result["paper"])
    print("Chunk ID    :", result["chunk_id"])
    print("Text       :", result["text"][:300].replace("\n", " "))


Rank       : 1
Score      : 0.7830531597137451
FAISS index: 297
Paper      : paper2
Chunk ID    : 40
Text       : unfamiliar with EHG tracings, integration into monitors and records is incomplete, and, as with any diagnostic test, false positives or negatives carry clinical and ethical costs; large-scale validation and training are prerequisites for routine use. 3. Electromyography (EMG) for Uterine Contractili

Rank       : 2
Score      : 0.7452520728111267
FAISS index: 301
Paper      : paper2
Chunk ID    : 44
Text       : invasive EMG is crucial for interpreting surface recordings and for developing models of uterine excitation and propagation. 3.2. Physiological Basis of Uterine EMG vs. Skeletal Muscle EMG Electromyography in the classical sense often refers to recording electrical activity from skeletal muscle. Ute

Rank       : 3
Score      : 0.7357030510902405
FAISS index: 311
Paper      : paper2
Chunk ID    : 54
Text       : systems could eventually provide a comprehensive asse

In [31]:
print(
    json.dumps(
        gold_queries[0],
        indent=2,
        ensure_ascii=False
    )
)

{
  "query_id": "Q001",
  "question": "What are the main characteristics of uterine electromyography (EMG) signals?",
  "category": "signal_characteristics",
  "relevant_paper_ids": [
    "paper1",
    "paper2",
    "paper3",
    "paper4",
    "paper5"
  ]
}


In [32]:
def get_unique_papers(
    results
):
    """
    Convert chunk-level retrieval results
    into ranked unique paper IDs.
    """

    seen = set()
    papers = []

    for result in results:

        paper = result["paper"]

        if paper not in seen:

            seen.add(paper)

            papers.append(paper)

    return papers

In [33]:
retrieved_papers = get_unique_papers(results)

print("Unique retrieved papers:")
print(retrieved_papers)

Unique retrieved papers:
['paper2']


In [34]:
def retrieve(
    query,
    chunk_top_k=20
):
    """
    Retrieve chunk_top_k chunks using MiniLM + FAISS.
    """

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        chunk_top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx == -1:
            continue

        row = metadata_df.iloc[idx]

        results.append({
            "chunk_rank": rank,
            "faiss_index": int(idx),
            "score": float(score),
            "paper": row["paper"],
            "chunk_id": int(row["chunk_id"]),
            "text": row["text"]
        })

    return results

In [35]:
def get_top_papers(
    results,
    top_k=5
):
    """
    Convert ranked chunk results into
    ranked unique paper results.
    """

    seen = set()
    papers = []

    for result in results:

        paper = result["paper"]

        if paper in seen:
            continue

        seen.add(paper)

        papers.append({
            "paper": paper,
            "score": result["score"],
            "chunk_id": result["chunk_id"],
            "chunk_rank": result["chunk_rank"]
        })

        if len(papers) >= top_k:
            break

    return papers

In [36]:
query = gold_queries[0]["question"]

chunk_results = retrieve(
    query,
    chunk_top_k=20
)

paper_results = get_top_papers(
    chunk_results,
    top_k=5
)

In [37]:
print("QUERY:")
print(query)

print("\n" + "=" * 80)
print("TOP-5 UNIQUE PAPERS")
print("=" * 80)

for result in paper_results:

    print(
        f"Rank {len([])+1}"
    )

QUERY:
What are the main characteristics of uterine electromyography (EMG) signals?

TOP-5 UNIQUE PAPERS
Rank 1
Rank 1
Rank 1


In [38]:
for rank, result in enumerate(
    paper_results,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        "Paper:",
        result["paper"]
    )

    print(
        "Score:",
        result["score"]
    )

    print(
        "Supporting chunk:",
        result["chunk_id"]
    )

    print(
        "Original chunk rank:",
        result["chunk_rank"]
    )


Rank 1
Paper: paper2
Score: 0.7830531597137451
Supporting chunk: 40
Original chunk rank: 1

Rank 2
Paper: paper1
Score: 0.7153208255767822
Supporting chunk: 194
Original chunk rank: 9

Rank 3
Paper: paper7
Score: 0.6993667483329773
Supporting chunk: 140
Original chunk rank: 10


In [39]:
gold_papers = gold_queries[0]["relevant_paper_ids"]

retrieved_papers = [
    result["paper"]
    for result in paper_results
]

print("Gold papers:")
print(gold_papers)

print("\nRetrieved papers:")
print(retrieved_papers)

print("\nIntersection:")
print(
    set(gold_papers) &
    set(retrieved_papers)
)

Gold papers:
['paper1', 'paper2', 'paper3', 'paper4', 'paper5']

Retrieved papers:
['paper2', 'paper1', 'paper7']

Intersection:
{'paper1', 'paper2'}


In [40]:
def hit_rate_at_k(
    retrieved_papers,
    relevant_papers,
    k=5
):
    """
    Returns 1 if at least one relevant paper
    appears in the top-k retrieved papers.
    """

    retrieved = retrieved_papers[:k]

    relevant = set(relevant_papers)

    return int(
        any(
            paper in relevant
            for paper in retrieved
        )
    )

In [41]:
hit = hit_rate_at_k(
    retrieved_papers,
    gold_papers,
    k=5
)

print("Hit Rate@5:", hit)

Hit Rate@5: 1


In [42]:
def ndcg_at_k(
    retrieved_papers,
    relevant_papers,
    k=5
):
    """
    Binary-relevance NDCG@k at the paper level.
    """

    retrieved = retrieved_papers[:k]

    relevant = set(relevant_papers)

    # Relevance vector
    relevance = [
        1 if paper in relevant else 0
        for paper in retrieved
    ]

    # DCG
    dcg = 0.0

    for rank, rel in enumerate(
        relevance,
        start=1
    ):

        if rel:
            dcg += (
                rel /
                np.log2(rank + 1)
            )

    # Ideal DCG
    num_relevant_in_top_k = min(
        len(relevant),
        k
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            num_relevant_in_top_k + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [43]:
ndcg = ndcg_at_k(
    retrieved_papers,
    gold_papers,
    k=5
)

print("NDCG@5:", ndcg)

NDCG@5: 0.5531464700081437


In [44]:
evaluation_results = []

for item in gold_queries:

    query_id = item.get("id")
    question = item["question"]

    gold_papers = item[
        "relevant_paper_ids"
    ]

    # Retrieve chunks
    chunk_results = retrieve(
        question,
        chunk_top_k=20
    )

    # Convert to top-5 unique papers
    paper_results = get_top_papers(
        chunk_results,
        top_k=5
    )

    retrieved_papers = [
        result["paper"]
        for result in paper_results
    ]

    # Metrics
    hit = hit_rate_at_k(
        retrieved_papers,
        gold_papers,
        k=5
    )

    ndcg = ndcg_at_k(
        retrieved_papers,
        gold_papers,
        k=5
    )

    evaluation_results.append({

        "query_id": query_id,

        "question": question,

        "gold_paper_ids": gold_papers,

        "retrieved_paper_ids":
            retrieved_papers,

        "hit_rate@5": hit,

        "ndcg@5": ndcg
    })

print(
    "Evaluated queries:",
    len(evaluation_results)
)

Evaluated queries: 30


In [45]:
eval_df = pd.DataFrame(
    evaluation_results
)

overall_hit_rate = (
    eval_df["hit_rate@5"].mean()
)

overall_ndcg = (
    eval_df["ndcg@5"].mean()
)

print("=" * 60)
print("MINILM + FAISS BASELINE")
print("=" * 60)

print(
    f"Queries    : {len(eval_df)}"
)

print(
    f"Hit Rate@5 : {overall_hit_rate:.4f}"
)

print(
    f"NDCG@5     : {overall_ndcg:.4f}"
)

print(
    f"Hit Rate@5 : {overall_hit_rate * 100:.2f}%"
)

print(
    f"NDCG@5     : {overall_ndcg * 100:.2f}%"
)

MINILM + FAISS BASELINE
Queries    : 30
Hit Rate@5 : 1.0000
NDCG@5     : 0.5714
Hit Rate@5 : 100.00%
NDCG@5     : 57.14%


In [46]:
display(
    eval_df[
        [
            "query_id",
            "question",
            "gold_paper_ids",
            "retrieved_paper_ids",
            "hit_rate@5",
            "ndcg@5"
        ]
    ]
)

,query_id,question,gold_paper_ids,retrieved_paper_ids,hit_rate@5,ndcg@5
0,None,What are the main characteristics of uterine e...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper1, paper7]",1,0.553146
1,None,What physiological activity generates uterine ...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper1, paper5, paper7]",1,0.722727
2,None,How do uterine EMG signals change as pregnancy...,"[paper1, paper2, paper3, paper4, paper5]","[paper7, paper2, paper1, paper4]",1,0.529635
3,None,How does uterine electrical activity change du...,"[paper1, paper2, paper3, paper4, paper5]","[paper1, paper4, paper7, paper2, paper5]",1,0.830420
4,None,What differences in uterine EMG activity have ...,"[paper1, paper2, paper3, paper4, paper5]","[paper7, paper1, paper2, paper4, paper5]",1,0.660840
5,None,What frequency components are commonly observe...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper1, paper6, paper10]",1,0.553146
6,None,What frequency bands have been used for analyz...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper1, paper6, paper10]",1,0.553146
7,None,What is the significance of the dominant or pe...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper10, paper1, paper7]",1,0.508740
8,None,How is power spectral density used to characte...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper10, paper7, paper6]",1,0.339160
9,None,What spectral features of uterine EMG have bee...,"[paper1, paper2, paper3, paper4, paper5]","[paper2, paper7, paper1, paper6, paper10]",1,0.508740


In [47]:
failed = eval_df[
    eval_df["hit_rate@5"] == 0
]

print(
    "Failed queries:",
    len(failed)
)

display(failed)

Failed queries: 0


,query_id,question,gold_paper_ids,retrieved_paper_ids,hit_rate@5,ndcg@5


In [48]:
baseline_eval_path = (
    EVALUATION_PATH /
    "minilm_faiss_paper_level_evaluation.json"
)

with open(
    baseline_eval_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    baseline_eval_path
)

Saved: /content/drive/MyDrive/uterine-emg-rag/evaluation/minilm_faiss_paper_level_evaluation.json


In [49]:
baseline_csv_path = (
    EVALUATION_PATH /
    "minilm_faiss_paper_level_evaluation.csv"
)

eval_df.to_csv(
    baseline_csv_path,
    index=False
)

print(
    "Saved:",
    baseline_csv_path
)

Saved: /content/drive/MyDrive/uterine-emg-rag/evaluation/minilm_faiss_paper_level_evaluation.csv


In [50]:
query = gold_queries[0]["question"]

chunk_results = retrieve(
    query,
    chunk_top_k=20
)

paper_results = get_top_papers(
    chunk_results,
    top_k=5
)

for rank, result in enumerate(
    paper_results,
    start=1
):
    print(
        f"Rank {rank}: "
        f"{result['paper']} | "
        f"score={result['score']:.4f} | "
        f"chunk={result['chunk_id']}"
    )

Rank 1: paper2 | score=0.7831 | chunk=40
Rank 2: paper1 | score=0.7153 | chunk=194
Rank 3: paper7 | score=0.6994 | chunk=140


In [51]:
print(
    "Gold papers:",
    gold_queries[0]["relevant_paper_ids"]
)

Gold papers: ['paper1', 'paper2', 'paper3', 'paper4', 'paper5']


In [52]:
print("=" * 80)
print("TOP-20 CHUNK RETRIEVAL")
print("=" * 80)

for result in chunk_results:

    print(
        f"Chunk rank={result['chunk_rank']:2d} | "
        f"paper={result['paper']:8s} | "
        f"chunk={result['chunk_id']:3d} | "
        f"score={result['score']:.4f}"
    )

TOP-20 CHUNK RETRIEVAL
Chunk rank= 1 | paper=paper2   | chunk= 40 | score=0.7831
Chunk rank= 2 | paper=paper2   | chunk= 44 | score=0.7453
Chunk rank= 3 | paper=paper2   | chunk= 54 | score=0.7357
Chunk rank= 4 | paper=paper2   | chunk= 52 | score=0.7293
Chunk rank= 5 | paper=paper2   | chunk= 49 | score=0.7285
Chunk rank= 6 | paper=paper2   | chunk= 25 | score=0.7244
Chunk rank= 7 | paper=paper2   | chunk= 41 | score=0.7244
Chunk rank= 8 | paper=paper2   | chunk= 15 | score=0.7184
Chunk rank= 9 | paper=paper1   | chunk=194 | score=0.7153
Chunk rank=10 | paper=paper7   | chunk=140 | score=0.6994
Chunk rank=11 | paper=paper2   | chunk= 42 | score=0.6988
Chunk rank=12 | paper=paper7   | chunk= 34 | score=0.6982
Chunk rank=13 | paper=paper2   | chunk= 17 | score=0.6974
Chunk rank=14 | paper=paper2   | chunk= 47 | score=0.6939
Chunk rank=15 | paper=paper2   | chunk= 43 | score=0.6899
Chunk rank=16 | paper=paper2   | chunk= 87 | score=0.6893
Chunk rank=17 | paper=paper2   | chunk= 50 | scor

In [53]:
def get_top_papers(results, top_k=5):
    """
    Convert ranked chunk results into ranked unique papers.
    """

    seen = set()
    papers = []

    for result in results:
        paper = result["paper"]

        if paper in seen:
            continue

        seen.add(paper)
        papers.append(result)

        if len(papers) == top_k:
            break

    return papers

In [54]:
chunk_results = retrieve(
    question,
    chunk_top_k=100
)

paper_results = get_top_papers(
    chunk_results,
    top_k=5
)

In [55]:
def hit_rate_at_5(retrieved_papers, relevant_papers):
    return int(
        len(set(retrieved_papers) & set(relevant_papers)) > 0
    )

In [56]:
def ndcg_at_5(retrieved_papers, relevant_papers):
    relevant = set(relevant_papers)

    dcg = 0.0
    for i, paper in enumerate(retrieved_papers[:5], start=1):
        if paper in relevant:
            dcg += 1 / np.log2(i + 1)

    ideal_hits = min(len(relevant), 5)
    idcg = sum(
        1 / np.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return 0.0 if idcg == 0 else dcg / idcg

In [57]:
evaluation_results = []

for item in gold_queries:

    question = item["question"]
    relevant = item["relevant_paper_ids"]

    chunk_results = retrieve(
        question,
        chunk_top_k=100
    )

    paper_results = get_top_papers(
        chunk_results,
        top_k=5
    )

    retrieved = [p["paper"] for p in paper_results]

    evaluation_results.append({
        "question": question,
        "gold_papers": relevant,
        "retrieved_papers": retrieved,
        "hit_rate@5": hit_rate_at_5(retrieved, relevant),
        "ndcg@5": ndcg_at_5(retrieved, relevant)
    })

In [58]:
eval_df = pd.DataFrame(evaluation_results)

print(f"Queries evaluated: {len(eval_df)}")
print(f"Average Hit Rate@5: {eval_df['hit_rate@5'].mean():.4f}")
print(f"Average NDCG@5: {eval_df['ndcg@5'].mean():.4f}")

Queries evaluated: 30
Average Hit Rate@5: 1.0000
Average NDCG@5: 0.5846


In [59]:
eval_df.to_csv(
    EVALUATION_PATH / "retrieval_evaluation.csv",
    index=False
)

with open(
    EVALUATION_PATH / "retrieval_evaluation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(evaluation_results, f, indent=2)

In [61]:
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

embedding_model = SentenceTransformer( "sentence-transformers/all-MiniLM-L6-v2" )
reranker = CrossEncoder( "cross-encoder/ms-marco-MiniLM-L-6-v2" )
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained( MODEL_NAME )
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
llm = AutoModelForCausalLM.from_pretrained( MODEL_NAME, quantization_config=quant_config, device_map="auto" )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [62]:
def build_prompt(question, contexts):
    context = "\n\n".join(
        [c["text"] for c in contexts]
    )

    return f"""You are an expert assistant for uterine electromyography research.

Answer the question using ONLY the information provided in the context.

Context:
{context}

Question:
{question}

Answer:"""

In [63]:
def generate_answer(question, contexts):

    prompt = build_prompt(question, contexts)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful biomedical research assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm.device)

    generated_ids = llm.generate(
        **model_inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=0.0
    )

    generated_ids = generated_ids[
        :,
        model_inputs.input_ids.shape[1]:
    ]

    answer = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    return answer.strip()

In [64]:
llm_results = []

for i, item in enumerate(gold_queries, start=1):

    question = item["question"]

    # Retrieve many chunks
    chunk_results = retrieve(
        question,
        chunk_top_k=100
    )

    # Rerank them
    pairs = [
        (question, chunk["text"])
        for chunk in chunk_results
    ]

    scores = reranker.predict(pairs)

    for chunk, score in zip(chunk_results, scores):
        chunk["rerank_score"] = float(score)

    reranked = sorted(
        chunk_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Use the top 5 reranked chunks
    top_chunks = reranked[:5]

    answer = generate_answer(
        question,
        top_chunks
    )

    llm_results.append({
        "query_number": i,
        "question": question,
        "gold_papers": item["relevant_paper_ids"],
        "retrieved_papers": [c["paper"] for c in top_chunks],
        "retrieved_chunk_ids": [c["chunk_id"] for c in top_chunks],
        "answer": answer
    })

    print(f"Completed {i}/{len(gold_queries)}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Completed 1/30
Completed 2/30
Completed 3/30
Completed 4/30
Completed 5/30
Completed 6/30
Completed 7/30
Completed 8/30
Completed 9/30
Completed 10/30
Completed 11/30
Completed 12/30
Completed 13/30
Completed 14/30
Completed 15/30
Completed 16/30
Completed 17/30
Completed 18/30
Completed 19/30
Completed 20/30
Completed 21/30
Completed 22/30
Completed 23/30
Completed 24/30
Completed 25/30
Completed 26/30
Completed 27/30
Completed 28/30
Completed 29/30
Completed 30/30


In [65]:
output_json = EVALUATION_PATH / "llm_answers.json"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(
        llm_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", output_json)

Saved: /content/drive/MyDrive/uterine-emg-rag/evaluation/llm_answers.json


In [66]:
output_txt = EVALUATION_PATH / "llm_answers.txt"

with open(output_txt, "w", encoding="utf-8") as f:

    for item in llm_results:

        f.write("=" * 100 + "\n")
        f.write(f"Query {item['query_number']}\n\n")

        f.write("Question:\n")
        f.write(item["question"] + "\n\n")

        f.write("Gold papers:\n")
        f.write(", ".join(item["gold_papers"]) + "\n\n")

        f.write("Retrieved papers:\n")
        f.write(", ".join(item["retrieved_papers"]) + "\n\n")

        f.write("Retrieved chunk IDs:\n")
        f.write(", ".join(map(str, item["retrieved_chunk_ids"])) + "\n\n")

        f.write("Answer:\n")
        f.write(item["answer"] + "\n\n")